# 密度泛函理论 / Density functional theory

---

密度泛函理论（Density Functional Theory）是一种研究多电子体系电子结构的量子力学方法。密度泛函理论在物理和化学上都有广泛的应用，特别是用来研究分子和凝聚态的性质，是凝聚态物理和计算化学领域最常用的方法之一。

Density functional theory (DFT) is a quantum mechanical method for studying the electronic structure of many-electron systems. DFT is widely used in physics and chemistry, particularly for studying the properties of molecules and condensed matter, and is one of the most commonly used methods in the fields of condensed matter physics and computational chemistry.

电子结构理论的经典方法，特别是Hartree-Fock方法和后Hartree-Fock方法，是基于复杂的多电子波函数的。密度泛函理论的主要目标就是用电子密度取代波函数做为研究的基本量。因为多电子波函数有3N个变量（N 为电子数，每个电子包含三个空间变量），而电子密度仅是三个变量的函数，无论在概念上还是实际上都更方便处理。

Classical methods of electronic structure theory, particularly the Hartree-Fock method and post-Hartree-Fock methods, are based on complicated many-electron wave functions. The main goal of DFT is to replace the wave function with the electron density as the basic quantity of study. Because the many-electron wave function has 3N variables (N being the number of electrons, each electron contributing three spatial variables), while the electron density is just a function of three variables, it is both conceptually and practically more convenient to work with.

Hohenberg-Kohn定理为泛函理论提供了坚实的理论基础：
1. 第一定理指出体系的基态能量仅仅是电子密度的泛函。
2. 第二定理证明了以基态密度为变量，将体系能量通过变分得到最小值之后就得到了基态能量。

The Hohenberg-Kohn theorems provide a solid theoretical foundation for density functional theory:
1. The first theorem states that the ground-state energy of a system is a functional solely of the electron density.
2. The second theorem proves that by taking the ground-state density as the variable and variationally minimizing the system's energy, one obtains the ground-state energy.

密度泛函理论最普遍的应用是通过Kohn-Sham方法实现的。在Kohn-Sham DFT框架中，最难处理的多体问题被简化成了一个没有相互作用的电子在有效势场中的运动问题。

The most common application of DFT is through the Kohn-Sham method. In the Kohn-Sham DFT framework, the most difficult many-body problem is reduced to the problem of non-interacting electrons moving in an effective potential field.

<!-- bilingual -->

# Numpy 1D-DFT
---

http://dcwww.camd.dtu.dk/~askhl/files/python-dft-exercises.pdf

Goal: write our own Kohn-Sham (KS) DFT code

Target: a harmonic oscillator including kinetic energy, electrostatic repulsion between the electrons, and the local density approximation for electronic interactions, ignoring correlation.

Hamiltonian:    
$$\hat{H}=-\frac{1}{2}\frac{d^2}{dx^2}+v(x)\\
v(x)=v_{Ha}(x)+v_{LDA}(x)+x^2$$

What we have to do?
1. Represent the Hamiltonian
2. Calculate the KS wavefunctions, the density

<!-- bilingual -->

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## Differential operator
---

In order to represent kinetic operator

<!-- bilingual -->

### First order differentiation

approximate:

$$(\frac{dy}{dx})_{i}=\frac{y_{i+1}-{y_{i}}}{h}$$

then:

$$D_{ij}=\frac{\delta_{i+1,j}-\delta_{i,j}}{h}$$

we could write as follows:

$$(\frac{dy}{dx})_{i}=D_{ij} y_{j}$$

The derivative may not be well defined at the end of the grid.

$\delta_{ij}$ is Kronecker delta

Einstein summation is used for last equation

<!-- bilingual -->

In [ ]:
n_grid=200
x=np.linspace(-5,5,n_grid)
h=x[1]-x[0]
D=-np.eye(n_grid)+np.diagflat(np.ones(n_grid-1),1)
D = D / h

### Second oorder differentiation

In the same way as the first order:

$$D^2_{ij}=\frac{\delta_{i+1,j}-2\delta_{i,j}+\delta_{i-1,j}}{h^2}$$

This could be written with the first order $D_{ij}$, as follows (take care of the transpose):

$$D^2_{ij}=-D_{ik}D_{jk}$$

The derivative may not be well defined at the end of the grid.

<!-- bilingual -->

In [ ]:
D2=D.dot(-D.T)
D2[-1,-1]=D2[0,0]

## Non-interacting electrons
---

This is the Hamiltonian of non-interacting free particles in a box given by the size of grid:

$$\hat{H} = \hat{T} = - \frac{1}{2} \frac{d^2}{dx^2}$$

<!-- bilingual -->

We could solve the KS equation as follows:

<!-- bilingual -->

In [ ]:
eig_non, psi_non=np.linalg.eigh(-D2/2)

plot (energies are shown in the label)

<!-- bilingual -->

In [ ]:
for i in range(3):
    plt.plot(x,psi_non[:,i], label=f"{eig_non[i]:.4f}")
    plt.legend(loc=1)

## Harmonic oscillator
---

include the external potential $v_{ext}=x^2$:
$$\hat{H} = \hat{T} = - \frac{1}{2} \frac{d^2}{dx^2} + x^2$$

we can write the potential as a matrix $X$, as follows:

<!-- bilingual -->

In [ ]:
X=np.diagflat(x*x)

and solve the KS.

<!-- bilingual -->

In [ ]:
eig_harm, psi_harm = np.linalg.eigh(-D2/2+X)

plot

<!-- bilingual -->

In [ ]:
for i in range(5):
    plt.plot(x,psi_harm[:,i], label=f"{eig_harm[i]:.4f}")
    plt.legend(loc=1)

## Density
---

We will want to include the Coulomb or Hatree interacion as well as LDA exchange

Both of which are density functinals

So we need to calculate the electron density

Each state should be normalized:
$$\int \lvert \psi \rvert ^2 dx = 1$$

let $f_n$ be occupation numbers, the density $n(x)$ can be written as follows:
$$n(x)=\sum_n f_n \lvert \psi(x) \rvert ^2 $$

Note:
1. Each state fits up to two electrons: one with spin up, and one with spin down.
2. In DFT, we calculate the ground state.

<!-- bilingual -->

In [ ]:
# integral
def integral(x,y,axis=0):
    dx=x[1]-x[0]
    return np.sum(y*dx, axis=axis)

number of electrons

<!-- bilingual -->

In [ ]:
num_electron=17

density

<!-- bilingual -->

In [ ]:
def get_nx(num_electron, psi, x):
    # normalization
    I=integral(x,psi**2,axis=0)
    normed_psi=psi/np.sqrt(I)[None, :]
    
    # occupation num
    fn=[2 for _ in range(num_electron//2)]
    if num_electron % 2:
        fn.append(1)

    # density
    res=np.zeros_like(normed_psi[:,0])
    for ne, psi  in zip(fn,normed_psi.T):
        res += ne*(psi**2)
    return res

plot

<!-- bilingual -->

In [ ]:
plt.plot(get_nx(num_electron,psi_non, x), label="non")
plt.plot(get_nx(num_electron,psi_harm, x), label="harm")
plt.legend(loc=1)

## Exchange energy
---

Consider the exchange functional in the LDA and ignore the correlation for simplicity:

$$ E_X^{LDA}[n] = -\frac{3}{4} \left(\frac{3}{\pi}\right)^{1/3} \int n^{4/3} dx$$

The potential is given by the derivative of the exchange energy with respect to the density:

$$ v_X^{LDA}[n] = \frac{\partial E_X^{LDA}}{\partial n} = - \left(\frac{3}{\pi}\right)^{1/3} n^{1/3}$$

<!-- bilingual -->

In [ ]:
def get_exchange(nx,x):
    energy=-3./4.*(3./np.pi)**(1./3.)*integral(x,nx**(4./3.))
    potential=-(3./np.pi)**(1./3.)*nx**(1./3.)
    return energy, potential

## coulomb potential
---

Electrostatic energy or Hatree energy

The expression of 3D-Hatree energy is not converged in 1D.

Hence we cheat and use a modified as follows:

$$ E_{Ha}=\frac{1}{2}\iint \frac{n(x)n(x')}{\sqrt{(x-x')^2+\varepsilon}}dxdx'$$

where $\varepsilon$ is a small positive constant

The potential is given by:
$$ v_{Ha}=\int \frac{n(x')}{\sqrt{(x-x')^2+\varepsilon}}dx'$$

In a matirx expression:
$$E_{Ha} = \frac{1}{2} \frac{n_in_jh^2}{\sqrt{(x_{i}-x_{j})^2+\varepsilon}}$$
$$v_{Ha, i} = \frac{n_jh}{\sqrt{(x_{i}-x_{j})^2+\varepsilon}}$$

<!-- bilingual -->

In [ ]:
def get_hatree(nx,x, eps=1e-1):
    h=x[1]-x[0]
    energy=np.sum(nx[None,:]*nx[:,None]*h**2/np.sqrt((x[None,:]-x[:,None])**2+eps)/2)
    potential=np.sum(nx[None,:]*h/np.sqrt((x[None,:]-x[:,None])**2+eps),axis=-1)
    return energy, potential

## Solve the KS equation：Self-consistency loop / Solve the KS equation: self-consistency loop

---

1. initialize the density (you can take an arbitrary constant)
2. Calculate the Exchange and Hatree potentials
3. Calculate the Hamiltonian
4. Calculate the wavefunctions and eigen values
5. If not converged, calculate the density and back to 1.

<!-- bilingual -->

In [ ]:
def print_log(i,log):
    print(f"step: {i:<5} energy: {log['energy'][-1]:<10.4f} energy_diff: {log['energy_diff'][-1]:.10f}")

max_iter=1000
energy_tolerance=1e-5
log={"energy":[float("inf")], "energy_diff":[float("inf")]}

nx=np.zeros(n_grid)
for i in range(max_iter):
    ex_energy, ex_potential=get_exchange(nx,x)
    ha_energy, ha_potential=get_hatree(nx,x)
    
    # Hamiltonian
    H=-D2/2+np.diagflat(ex_potential+ha_potential+x*x)
    
    energy, psi= np.linalg.eigh(H)
    
    # log
    log["energy"].append(energy[0])
    energy_diff=energy[0]-log["energy"][-2]
    log["energy_diff"].append(energy_diff)
    print_log(i,log)
    
    # convergence
    if abs(energy_diff) < energy_tolerance:
        print("converged!")
        break
    
    # update density
    nx=get_nx(num_electron,psi,x)
else:
    print("not converged")

plot

<!-- bilingual -->

In [ ]:
for i in range(5):
    plt.plot(x,psi[:,i], label=f"{energy[i]:.4f}")
    plt.legend(loc=1)

compare the density to free particles

<!-- bilingual -->

In [ ]:
plt.plot(nx)
plt.plot(get_nx(num_electron,psi_harm,x), label="no-interaction")
plt.legend()